In [24]:
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.decomposition import PCA
from inmoose.pycombat import pycombat_norm
import os

# ── Configuration ─────────────────────────────────────────────────────────────
ALL_DATASETS = ['GSE27097', 'GSE40279', 'GSE41037', 'GSE42861', "GSE84727", "GSE111629", "GSE157131", "GSE197674"]

# Select which datasets to include in the combined output
# Change this list to select a subset e.g. ['GSE27097', 'GSE40279']
#DATASETS_TO_COMBINE = ['GSE27097', 'GSE40279', 'GSE42861']
DATASETS_TO_COMBINE = ALL_DATASETS

# Set to True to save each dataset individually to 03_final
SAVE_INDIVIDUAL = True

# ── Load data ─────────────────────────────────────────────────────────────────
cpgs = pd.read_csv('/work/gr-fe/bryan/EpiHydra/data/03_final/methylgpt/probe_ids_type3.csv')['illumina_probe_id']

cpg_list = []
meta_list = []
max_i = 0
best_i = 0

for i, data in enumerate(ALL_DATASETS):
    filepath = f'/work/gr-fe/bryan/EpiHydra/data/02_processed/{data}/methylation.pkl'

    with open(filepath, 'rb') as file:
        methylation = pickle.load(file)

    cpg_list.append(methylation['CpG'].reindex(cpgs))
    methylation['meta']['Dataset'] = [data for _ in range(methylation['meta'].shape[0])]
    meta_list.append(methylation['meta'])
    del methylation

    if cpg_list[i].shape[1] > max_i:
        max_i = cpg_list[i].shape[1]
        best_i = i

print(f"Reference dataset (best_i={best_i}): {ALL_DATASETS[best_i]}")

# ── Helper plot function ──────────────────────────────────────────────────────
def plot_pca(X, meta_df, title):
    X_filled = X.fillna(X.median()).dropna()
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_filled.T)

    datasets = meta_df['Dataset'].unique()
    colours = cm.tab10(np.linspace(0, 1, len(datasets)))
    colour_map = dict(zip(datasets, colours))

    fig, ax = plt.subplots(figsize=(8, 6))

    for dataset in datasets:
        mask = meta_df['Dataset'] == dataset
        ax.scatter(
            X_pca[mask, 0],
            X_pca[mask, 1],
            label=dataset,
            color=colour_map[dataset],
            alpha=0.7
        )

    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
    ax.set_title(title)
    ax.legend(title='Dataset', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

# ── Pre-correction PCA ────────────────────────────────────────────────────────
combined_pre = pd.concat(cpg_list, axis=1)
combined_meta_pre = pd.concat(meta_list)
plot_pca(combined_pre, combined_meta_pre, 'PCA by Dataset (Pre-correction)')
del combined_pre, combined_meta_pre

# ── Pairwise batch correction to reference ────────────────────────────────────
ref_cpg  = cpg_list[best_i].copy()
ref_meta = meta_list[best_i].copy()
ref_meta['Dataset'] = ALL_DATASETS[best_i]

# Store all corrected datasets keyed by name for easy access later
corrected_store = {ALL_DATASETS[best_i]: {'CpG': ref_cpg, 'meta': ref_meta}}

for i, (cpg_df, meta_df) in enumerate(zip(cpg_list, meta_list)):
    if i == best_i:
        continue

    # ── Find CpGs with no NaN in EITHER dataset — required for pycombat ───────
    valid_cpgs_ref = ref_cpg.index[~ref_cpg.isna().any(axis=1)]
    valid_cpgs_ds  = cpg_df.index[~cpg_df.isna().any(axis=1)]
    valid_cpgs = list(set(valid_cpgs_ref) & set(valid_cpgs_ds))

    print(f"{ALL_DATASETS[i]}: {len(valid_cpgs):,} CpGs valid for correction "
          f"({len(valid_cpgs)/len(ref_cpg.index)*100:.1f}% of all CpGs)")

    # ── Subset both datasets to NaN-free CpGs ─────────────────────────────────
    ref_subset = ref_cpg.loc[valid_cpgs]
    ds_subset  = cpg_df.loc[valid_cpgs]

    # ── Combine and build batch vector ────────────────────────────────────────
    combined = pd.concat([ref_subset, ds_subset], axis=1)
    batch = [0] * ref_subset.shape[1] + [1] * ds_subset.shape[1]

    # ── Age covariate ─────────────────────────────────────────────────────────
    covar_age = pd.DataFrame(
        {'age': np.concatenate([ref_meta['age'].values, meta_df['age'].values])},
        index=combined.columns
    )

    # ── Correct — ref_batch=0 keeps reference samples unchanged ───────────────
    corrected = pycombat_norm(
        combined,
        batch,
        ref_batch=0,
        covar_mod=covar_age,
        na_cov_action='remove'
    )

    # ── Extract corrected target samples only ─────────────────────────────────
    corrected_ds = corrected.iloc[:, ref_subset.shape[1]:]

    # ── Place corrected values into full reference CpG space ──────────────────
    corrected_full = pd.DataFrame(
        np.nan,
        index=ref_cpg.index,
        columns=corrected_ds.columns
    )
    corrected_full.loc[valid_cpgs] = corrected_ds.values

    meta_df = meta_df[meta_df['ID'].isin(corrected_full.columns)]
    corrected_store[ALL_DATASETS[i]] = {'CpG': corrected_full, 'meta': meta_df}

# ── Save individual datasets to 03_final ─────────────────────────────────────
if SAVE_INDIVIDUAL:
    for dataset_name, data in corrected_store.items():
        out_dir = f'/work/gr-fe/bryan/EpiHydra/data/03_final/{dataset_name}'
        os.makedirs(out_dir, exist_ok=True)
        out_path = f'{out_dir}/methylation.pkl'

        with open(out_path, 'wb') as file:
            pickle.dump(data, file)

        print(f"Saved {dataset_name} → {out_path} "
              f"({data['CpG'].shape[1]} samples, {data['CpG'].shape[0]} CpGs)")

# ── Build combined from selected subset ───────────────────────────────────────
print(f"\nBuilding combined dataset from: {DATASETS_TO_COMBINE}")

missing = [d for d in DATASETS_TO_COMBINE if d not in corrected_store]
if missing:
    raise ValueError(f"These datasets were not corrected and cannot be combined: {missing}")

combined_cpg  = pd.concat([corrected_store[d]['CpG']  for d in DATASETS_TO_COMBINE], axis=1)
combined_meta = pd.concat([corrected_store[d]['meta'] for d in DATASETS_TO_COMBINE])

print(f"Final combined matrix shape: {combined_cpg.shape}")
print(f"Missing values: {combined_cpg.isna().sum().sum():,} "
      f"({combined_cpg.isna().mean().mean()*100:.1f}%)")

# ── Post-correction PCA ───────────────────────────────────────────────────────
plot_pca(combined_cpg, combined_meta, 'PCA by Dataset (Pairwise correction to reference)')

# ── Save combined ─────────────────────────────────────────────────────────────
combined_df = {
    'CpG': combined_cpg,
    'meta': combined_meta
}

os.makedirs('/work/gr-fe/bryan/EpiHydra/data/02_processed/combined', exist_ok=True)

with open('/work/gr-fe/bryan/EpiHydra/data/02_processed/combined/methylation.pkl', 'wb') as file:
    pickle.dump(combined_df, file)

print("\nCombined dataset saved successfully.")


Reference dataset (best_i=7): GSE197674


ValueError: at least one array or dtype is required

In [27]:
tst = combined_pre.fillna(combined_pre.median())

In [28]:
tst

,GSM666344,GSM666345,GSM666350,GSM666351,GSM666352,GSM666353,GSM666354,GSM666355,GSM666357,GSM666361,...,GSM5927266,GSM5927267,GSM5927268,GSM5927269,GSM5927270,GSM5927271,GSM5927272,GSM5927273,GSM5927274,GSM5927275
illumina_probe_id,,,,,,,,,,,,,,,,,,,,,
cg00000109,0.06831,0.05091,0.05058,0.05382,0.05149,0.04106,0.03644,0.03587,0.05091,0.04736,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
cg00000292,0.83445,0.82689,0.82479,0.86973,0.88127,0.84240,0.85900,0.81303,0.81637,0.83030,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
cg00002033,0.06831,0.05091,0.05058,0.05382,0.05149,0.04106,0.03644,0.03587,0.05091,0.04736,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
cg00002426,0.81105,0.87160,0.89546,0.89960,0.85116,0.83261,0.86198,0.87273,0.86802,0.87935,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
cg00002719,0.06831,0.05091,0.05058,0.05382,0.05149,0.04106,0.03644,0.03587,0.05091,0.04736,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ch.6.20366279F,0.06831,0.05091,0.05058,0.05382,0.05149,0.04106,0.03644,0.03587,0.05091,0.04736,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ch.6.2925136R,0.06831,0.05091,0.05058,0.05382,0.05149,0.04106,0.03644,0.03587,0.05091,0.04736,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ch.7.3189261R,0.06831,0.05091,0.05058,0.05382,0.05149,0.04106,0.03644,0.03587,0.05091,0.04736,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [34]:

tst.dropna(axis=1)

,GSM666344,GSM666345,GSM666350,GSM666351,GSM666352,GSM666353,GSM666354,GSM666355,GSM666357,GSM666361,...,GSM1052204,GSM1052205,GSM1052206,GSM1052207,GSM1052208,GSM1052209,GSM1052210,GSM1052211,GSM1052212,GSM1052213
illumina_probe_id,,,,,,,,,,,,,,,,,,,,,
cg00000109,0.06831,0.05091,0.05058,0.05382,0.05149,0.04106,0.03644,0.03587,0.05091,0.04736,...,0.820480,0.780908,0.796633,0.768051,0.802091,0.813071,0.468836,0.826276,0.774397,0.839801
cg00000292,0.83445,0.82689,0.82479,0.86973,0.88127,0.84240,0.85900,0.81303,0.81637,0.83030,...,0.818195,0.802515,0.790492,0.811943,0.804485,0.801524,0.866637,0.797516,0.819985,0.786583
cg00002033,0.06831,0.05091,0.05058,0.05382,0.05149,0.04106,0.03644,0.03587,0.05091,0.04736,...,0.876199,0.719166,0.955019,0.799507,0.833903,0.840323,0.748935,0.834252,0.849573,0.674009
cg00002426,0.81105,0.87160,0.89546,0.89960,0.85116,0.83261,0.86198,0.87273,0.86802,0.87935,...,0.882978,0.864691,0.842286,0.844942,0.871696,0.838823,0.878334,0.838458,0.857438,0.822168
cg00002719,0.06831,0.05091,0.05058,0.05382,0.05149,0.04106,0.03644,0.03587,0.05091,0.04736,...,0.017385,0.042056,0.051514,0.051228,0.034950,0.035154,0.093209,0.051112,0.052688,0.034655
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ch.6.20366279F,0.06831,0.05091,0.05058,0.05382,0.05149,0.04106,0.03644,0.03587,0.05091,0.04736,...,0.054922,0.071049,0.106811,0.112229,0.049506,0.055788,0.162655,0.055435,0.084334,0.080524
ch.6.2925136R,0.06831,0.05091,0.05058,0.05382,0.05149,0.04106,0.03644,0.03587,0.05091,0.04736,...,0.049786,0.090068,0.073780,0.098159,0.038944,0.036655,0.144560,0.058208,0.056882,0.089678
ch.7.3189261R,0.06831,0.05091,0.05058,0.05382,0.05149,0.04106,0.03644,0.03587,0.05091,0.04736,...,0.057618,0.087853,0.072232,0.106762,0.058836,0.062347,0.079236,0.085398,0.066882,0.093773


In [51]:
cpg_list[2].dropna()

,GSM1007129,GSM1007130,GSM1007131,GSM1007132,GSM1007133,GSM1007134,GSM1007135,GSM1007136,GSM1007137,GSM1007138,...,GSM1007839,GSM1007840,GSM1007841,GSM1007842,GSM1007843,GSM1007844,GSM1007845,GSM1007846,GSM1007847,GSM1007848
illumina_probe_id,,,,,,,,,,,,,,,,,,,,,
cg00000292,0.826452,0.786005,0.821569,0.800461,0.846929,0.861608,0.681818,0.784737,0.826786,0.746127,...,0.800676,0.782873,0.675258,0.789442,0.629310,0.756856,0.753206,0.743481,0.739643,0.773988
cg00002426,0.664997,0.776752,0.718688,0.773340,0.760058,0.794964,0.723584,0.742225,0.746415,0.736423,...,0.783468,0.682987,0.792201,0.793917,0.755901,0.771837,0.787029,0.791851,0.808165,0.816977
cg00003994,0.044991,0.065413,0.037126,0.047140,0.031929,0.057937,0.068399,0.057447,0.064459,0.048139,...,0.069613,0.080460,0.064493,0.056075,0.087791,0.062281,0.081177,0.064447,0.060761,0.061161
cg00006414,0.077353,0.091298,0.080609,0.107725,0.130279,0.098889,0.117026,0.093538,0.122043,0.089337,...,0.095717,0.115239,0.094053,0.063734,0.074883,0.066782,0.072243,0.070335,0.075786,0.077058
cg00007981,0.096471,0.062367,0.046132,0.065258,0.057724,0.071634,0.074633,0.035222,0.101101,0.087379,...,0.063739,0.085328,0.091458,0.072285,0.088689,0.060973,0.049684,0.081715,0.072845,0.063445
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
cg27657283,0.081038,0.111551,0.101114,0.070997,0.097285,0.066596,0.096225,0.068592,0.075609,0.057486,...,0.102398,0.152743,0.099083,0.082945,0.108096,0.091549,0.079614,0.094235,0.098719,0.090517
cg27661264,0.197167,0.377573,0.272153,0.341921,0.270388,0.222771,0.363886,0.362355,0.351474,0.272481,...,0.246903,0.306851,0.239522,0.308926,0.236997,0.289531,0.294740,0.305630,0.351106,0.275140
cg27662379,0.034734,0.039018,0.058072,0.049498,0.029971,0.038705,0.026374,0.056011,0.028895,0.029263,...,0.029665,0.029819,0.037893,0.027946,0.031760,0.028757,0.024727,0.032984,0.026986,0.031913


In [48]:
with open(filepath, 'rb') as file:
    methylation = pickle.load(file)

In [50]:
methylation['CpG']

,GSM5925138,GSM5925139,GSM5925140,GSM5925141,GSM5925142,GSM5925143,GSM5925144,GSM5925145,GSM5925146,GSM5925147,...,GSM5927266,GSM5927267,GSM5927268,GSM5927269,GSM5927270,GSM5927271,GSM5927272,GSM5927273,GSM5927274,GSM5927275
ID_REF,,,,,,,,,,,,,,,,,,,,,
